# Pancreas FlowMap Gene Evaluation vs DEG

Standalone extraction notebook for Fig. 3 pancreas gene lists. It computes FlowMap gene-evaluation hits per cell type, DEG hits per cell type, and overlap tables for direct comparison.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import scanpy as sc
import scvelo as scv
import numpy as np

adata = sc.read_h5ad("./data/pancreas/pancreas_inferred_velocity.h5ad")
n_pcs = 50

# Use the matrix YOU trust (usually adata.X or adata.layers["Ms"])
X = adata.X.toarray() if hasattr(adata.X, "toarray") else adata.X

# Optional: standardize (recommended if you want PCA to behave)
X = StandardScaler(with_mean=True, with_std=True).fit_transform(X)

# Optional: "stretch PCA" (identity by default; customize if needed)
stretch = np.ones(X.shape[1])   # <- replace with gene-wise stretch if desired
X = X * stretch

# PCA
pca = PCA(n_components=n_pcs, svd_solver="arpack", random_state=0)
X_pca = pca.fit_transform(X)

# --------------------------
# OVERWRITE PCA IN ADATA
# --------------------------
adata.obsm["X_pca"] = X_pca
adata.varm["PCs"] = pca.components_.T
adata.uns["pca"] = {
    "variance": pca.explained_variance_,
    "variance_ratio": pca.explained_variance_ratio_,
}

print("✅ Overwrote PCA:", adata.obsm["X_pca"].shape)

## Load Data And Recreate Fig. 3 Embedding

In [ ]:
scv.tl.velocity(adata, mode="stochastic", vkey="stochastic_velocity")
scv.tl.velocity_graph(adata, vkey="stochastic_velocity")

scv.tl.velocity(adata, mode="dynamical", vkey="dynamical_velocity")
scv.tl.velocity_graph(adata, vkey="dynamical_velocity")

if "stochastic_velocity_pca" not in adata.obsm:
    print("Computing stochastic_velocity_pca...")
    scv.tl.velocity_embedding(adata, basis="pca", vkey="stochastic_velocity")

V_pca_stoch = adata.obsm["stochastic_velocity_pca"]

# Compute ONLY if missing
if "dynamical_velocity_pca" not in adata.obsm:
    print("Computing dynamical_velocity_pca...")
    scv.tl.velocity_embedding(adata, basis="pca", vkey="dynamical_velocity")

In [ ]:
X = adata.obsm["X_pca"]
V = adata.obsm["stochastic_velocity_pca"]
X_umap = adata.obsm["X_umap"]
cell_type = adata.obs["clusters"]

emb = VectorFieldEmbedder(
    X,
    V,
    dist_method="phase",
    dof=30,
    X_emb=X_umap,
    embed_kwargs={"n_neighbors": 30, "min_dist": 0.5, "spread": 1.0},
)
emb.fit_embedding(seed=RANDOM_STATE)

X_gene = adata.layers["spliced"]
V_gene = adata.layers["stochastic_velocity"]

if hasattr(X_gene, "toarray"):
    X_gene = X_gene.toarray()
if hasattr(V_gene, "toarray"):
    V_gene = V_gene.toarray()

emb.fit_gene_level_splines(
    dof_gene=30,
    dof_vf_gene=30,
    X=X_gene,
    V=V_gene,
)


## FlowMap Gene Evaluation Hits Per Cell Type

In [ ]:
genes = adata.var_names.to_numpy()
labels = adata.obs[CLUSTER_KEY].astype(str).to_numpy()
cell_types = list(adata.obs[CLUSTER_KEY].cat.categories)

evaluation_tables = []
results_by_cell_type = {}

for ct in cell_types:
    cell_idx = np.where(labels == ct)[0]
    print(f"Evaluating {ct}: {len(cell_idx)} cells")

    if len(cell_idx) < MIN_CELLS:
        print(f"  skip: fewer than {MIN_CELLS} cells")
        continue

    evaluator = SplineFitEvaluator(emb, mode="gene")
    res = evaluator.evaluate(cell_idx=cell_idx)
    results_by_cell_type[ct] = {"n_cells": len(cell_idx), "eval": res}

    expr_corr = np.asarray(res["expr_corr_gene"], dtype=float)
    vel_corr = np.asarray(res["vel_corr_gene"], dtype=float)
    score = expr_corr * vel_corr

    df_ct = pd.DataFrame({
        "cell_type": ct,
        "n_cells": len(cell_idx),
        "gene": genes,
        "expr_corr": expr_corr,
        "vel_corr": vel_corr,
        "evaluation_score": score,
    })
    df_ct = df_ct[np.isfinite(df_ct["evaluation_score"])].copy()
    df_ct = df_ct.sort_values("evaluation_score", ascending=False)
    df_ct["evaluation_rank"] = np.arange(1, len(df_ct) + 1)
    evaluation_tables.append(df_ct)

evaluation_genes = pd.concat(evaluation_tables, ignore_index=True)
evaluation_genes.to_csv(OUTDIR / "pancreas_flowmap_gene_evaluation_by_cell_type.csv", index=False)

top_evaluation_genes = evaluation_genes.query("evaluation_rank <= @TOP_N").copy()
top_evaluation_genes.to_csv(OUTDIR / f"pancreas_flowmap_top{TOP_N}_genes_by_cell_type.csv", index=False)

top_evaluation_genes.head(20)


## DEG Hits Per Cell Type

In [ ]:
sc.tl.rank_genes_groups(
    adata,
    groupby=CLUSTER_KEY,
    method="wilcoxon",
    use_raw=False,
    pts=True,
)

deg_tables = []

for ct in cell_types:
    df_ct = sc.get.rank_genes_groups_df(adata, group=ct).copy()
    df_ct = df_ct.rename(columns={"names": "gene"})
    df_ct.insert(0, "cell_type", ct)
    df_ct = df_ct.sort_values(["pvals_adj", "pvals", "scores"], ascending=[True, True, False], na_position="last")
    df_ct["deg_rank"] = np.arange(1, len(df_ct) + 1)
    deg_tables.append(df_ct)

deg_genes = pd.concat(deg_tables, ignore_index=True)
deg_genes.to_csv(OUTDIR / "pancreas_deg_by_cell_type.csv", index=False)

top_deg_genes = deg_genes.query("deg_rank <= @TOP_N").copy()
top_deg_genes.to_csv(OUTDIR / f"pancreas_deg_top{TOP_N}_genes_by_cell_type.csv", index=False)

top_deg_genes.head(20)


## Compare FlowMap Evaluation Genes Against DEG Genes

In [ ]:
overlap_rows = []
comparison_rows = []

for ct in cell_types:
    eval_ct = evaluation_genes[evaluation_genes["cell_type"] == ct].copy()
    deg_ct = deg_genes[deg_genes["cell_type"] == ct].copy()

    eval_top = set(eval_ct.head(TOP_N)["gene"])
    deg_top = set(deg_ct.head(TOP_N)["gene"])
    overlap = sorted(eval_top & deg_top)

    overlap_rows.append({
        "cell_type": ct,
        "top_n": TOP_N,
        "n_flowmap_top": len(eval_top),
        "n_deg_top": len(deg_top),
        "n_overlap": len(overlap),
        "jaccard": len(overlap) / len(eval_top | deg_top) if len(eval_top | deg_top) else np.nan,
        "overlap_genes": ",".join(overlap),
    })

    eval_keep = eval_ct[["gene", "evaluation_rank", "evaluation_score", "expr_corr", "vel_corr"]]
    deg_keep_cols = [c for c in ["gene", "deg_rank", "scores", "logfoldchanges", "pvals", "pvals_adj", "pct_nz_group", "pct_nz_reference"] if c in deg_ct.columns]
    deg_keep = deg_ct[deg_keep_cols]

    merged = eval_keep.merge(deg_keep, on="gene", how="outer")
    merged.insert(0, "cell_type", ct)
    merged["in_flowmap_top"] = merged["evaluation_rank"] <= TOP_N
    merged["in_deg_top"] = merged["deg_rank"] <= TOP_N
    merged["in_both_top"] = merged["in_flowmap_top"] & merged["in_deg_top"]
    merged = merged.sort_values(["in_both_top", "evaluation_rank", "deg_rank"], ascending=[False, True, True], na_position="last")
    comparison_rows.append(merged)

overlap_summary = pd.DataFrame(overlap_rows)
overlap_summary.to_csv(OUTDIR / f"pancreas_flowmap_deg_top{TOP_N}_overlap_summary.csv", index=False)

gene_comparison = pd.concat(comparison_rows, ignore_index=True)
gene_comparison.to_csv(OUTDIR / "pancreas_flowmap_vs_deg_gene_comparison.csv", index=False)

overlap_summary


In [ ]:
for ct in cell_types:
    print(f"\n=== {ct} ===")
    print("FlowMap top genes:")
    print(", ".join(top_evaluation_genes[top_evaluation_genes["cell_type"] == ct].head(20)["gene"]))
    print("DEG top genes:")
    print(", ".join(top_deg_genes[top_deg_genes["cell_type"] == ct].head(20)["gene"]))
    print("Overlap:")
    row = overlap_summary[overlap_summary["cell_type"] == ct]
    print(row["overlap_genes"].iloc[0] if len(row) else "")

print("\nSaved tables to:", OUTDIR.resolve())
